In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so
from PIL import Image
import numpy as np
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import icd10
import hvplot.pandas
import holoviews as hv
from bokeh.plotting import show
from pycirclize import Circos
from pycirclize.utils import load_example_tree_file
from workspace_dani.data_preprocessing.utils import data_handling

hv.extension("bokeh")

# Investigating different the other types of shock with ICD-10 code R57.x 
These ICD codes contain shock of types, cardiogenic(R57.0), hypovolaemic (R57.1), septic (R57.2), misc (including endotoxic) (57.8), unspecified (R57.9) after ICD-10-GM-2024



In [ ]:
project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")

dataset_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("build") / Path(
    "2024-03-26") / Path("build_dataset_cats.parquet")
dataset_path


In [ ]:
df_cats = data_handling.load_parquet_table(parquet_path=dataset_path, debug_mode=False)
df_cats

In [ ]:
admission_source_ids_path = project_root_path / Path(
    "intermediary_data/patient_unit_stay_ids/cohorts/admit_source_unit_type/included_ids_admit_source_unit_type.parquet")
admission_source_ids_path

In [ ]:
admission_dx_ids_path = project_root_path / Path(
    "intermediary_data/patient_unit_stay_ids/cohorts/admit_dx_path/included_ids_admit_dx_path.parquet")
admission_dx_ids_path

In [ ]:
cardiovascular_treatment_ids_path = project_root_path / Path(
    "intermediary_data/patient_unit_stay_ids/cohorts/cardiovascular_treatment/included_treatmentids.parquet")
cardiovascular_treatment_ids_path

In [ ]:
admission_source_inclusion_ids_df: pd.DataFrame = data_handling.load_parquet_table(parquet_path=admission_source_ids_path, debug_mode=False)

admission_dx_inclusion_ids_df: pd.DataFrame = data_handling.load_parquet_table(parquet_path=admission_dx_ids_path, debug_mode=False)

cardiovascular_treatment_inclusion_ids_df: pd.DataFrame = data_handling.load_parquet_table(parquet_path=cardiovascular_treatment_ids_path, debug_mode=False)


In [ ]:
diagnosis_parquet_path: Path = project_root_path / Path(
    "intermediary_data/data_cleanup/basic_cleanup/2024-03-26/diagnosis.parquet")
df_diagnosis_table = data_handling.load_parquet_table(parquet_path=diagnosis_parquet_path, debug_mode=False)
df_diagnosis_table


In [ ]:
stay_diagnosis: pd.Series = df_diagnosis_table[["patientunitstayid", "diagnosisstring", "icd9code"]]


In [ ]:
shock_cs_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("r57.0", case=False)
shock_hypoval_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("R57.1", case=False)
shock_septic_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("R57.2", case=False)
shock_misc_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("R57.8", case=False)
shock_nonspec_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("R57.9", case=False)

shock_dx_name_condition: pd.Series = stay_diagnosis["diagnosisstring"].str.contains("shock", case=False)

all_shock_conditions = shock_dx_name_condition | shock_cs_icd_code_condition | shock_hypoval_icd_code_condition | shock_septic_icd_code_condition | shock_misc_icd_code_condition | shock_nonspec_icd_code_condition

In [ ]:
stays_shock_all: pd.DataFrame = stay_diagnosis[all_shock_conditions]

### Number of unique patients that suffered from shock at some point:

In [ ]:
stay_diagnosis["patientunitstayid"].unique()

In [ ]:

stays_shock_all["patientunitstayid"].unique()